In [ ]:
import os
import pandas as pd
from tqdm import tqdm

In [ ]:
path = 'token_data.tsv'
df = pd.read_csv(path, sep='\t')

In [ ]:
len(df), df.columns.tolist()

In [ ]:
df['group'] = df.apply(lambda row: f"lemma={row['lemma']}, pos={row['pos']}", axis=1)
print(len(df['group'].unique()))
print(len(df['group_hash'].unique()))

In [ ]:
sentence_df = pd.read_csv('sentence_dataframe.tsv', sep='\t')
len(sentence_df), sentence_df.columns

In [ ]:
sentence_df.set_index('hash', inplace=True, drop=True)

In [ ]:
group_pct_dict = {}
group_hash_to_str = {}
group_hashes = list(df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    row = df[df['group_hash'] == group_hash].iloc[0]
    group_pct_dict[group_hash] = row['group_pct']
    group_hash_to_str[group_hash] = row['group']

In [ ]:
all_groups = list(df['group_hash'].unique())
all_groups = sorted(all_groups, key=lambda x: group_pct_dict[x], reverse=True)
[group_hash_to_str[hash] for hash in all_groups[:10]]

In [ ]:
N = 200
top_groups = all_groups[:N]
reduced_df = df[df['group_hash'].isin(top_groups)]
len(reduced_df)

In [ ]:
reduced_df.columns

In [ ]:
save_dir = 'dataframes'

In [ ]:
import uuid
# fix group hashes
# Group by lemma so cliticized forms land in the base verb bucket
def get_group_hash(lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{lemma}-{pos}"))
reduced_df['group_hash'] = reduced_df.apply(lambda row: get_group_hash(row['lemma'], row['pos']), axis=1)

In [ ]:
# reduced_df['group_str'] = reduced_df.apply(lambda row: f"LEMMA={row['lemma']}_POS={row['pos']}_PARTS_LEMMA={row['parts_lemma']}_PARTS_POS={row['parts_pos']}", axis=1)
# reduced_df['group_str'].value_counts()

In [ ]:
group_hashes = list(reduced_df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    group_df = reduced_df[reduced_df['group_hash'] == group_hash].copy()
    if group_df['lemma'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    if group_df['pos'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    assert(group_df['lemma'].nunique() == 1)
    assert(group_df['pos'].nunique() == 1)
    pos = group_df.iloc[0]['pos']
    lemma = group_df.iloc[0]['lemma']

    for i, row in group_df.iterrows():
        sentence_hashes = eval(row['sentences'])
        assert(type(sentence_hashes) is list)
        for j in range(3):
            if len(sentence_hashes) > j:
                sentence_hash = sentence_hashes[j]
                sentence_it = sentence_df.loc[sentence_hash, 'text_it']
                sentence_en = sentence_df.loc[sentence_hash, 'text_en']
                group_df.loc[i, f'sentence_{j+1}_it'] = sentence_it
                group_df.loc[i, f'sentence_{j+1}_en'] = sentence_en
    group_df['translation_en'] = ''
    group_df.drop(columns=['sentences'], inplace=True)
    save_path = os.path.join(save_dir, pos, f"{lemma}.tsv")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    cols = [
        'text', 'lemma', 'pos', 'translation_en', 'token_count', 'token_pct', 'group_count',
        'group_pct', 'parts_lemma', 'parts_pos', 'xpos', 'deprel', 'feats', 'token_hash', 'group_hash',
        'group', 'sentence_1_it', 'sentence_1_en', 'sentence_2_it',
        'sentence_2_en', 'sentence_3_it', 'sentence_3_en'
    ]
    missing_cols = list(set(cols).difference(set(group_df.columns)))
    for missing_col in missing_cols:
        assert('sentence_' in missing_col)
        group_df[missing_col] = ''
    assert(set(cols) == set(group_df.columns))
    group_df = group_df[cols]
    for i, row in group_df.iterrows():
        for col in cols:
            if type(row[col]) == str:
                group_df.loc[i, col] = row[col].strip()

    group_df.to_csv(save_path, sep='\t', index=False)